# 🩺 Chronic Kidney Disease (CKD) — ML Classification Pipeline

**Dataset:** 1659 patients × 54 features  
**Task:** Binary classification — CKD (1) vs No CKD (0)  
**Models:** Logistic Regression, Random Forest, SVM, XGBoost  
**Key challenge:** Severe class imbalance (≈ 11:1) — addressed with SMOTE

---

## Step 1 — Install Dependencies & Import Libraries

In [ ]:
# Install required packages
!pip install imbalanced-learn xgboost --quiet
print("✅ Packages installed")

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ── XGBoost ──────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── Imbalanced-learn ─────────────────────────────────────────────────────────
# NOTE: two of our 11 features (UrinaryTractInfections, FamilyHistoryKidneyDisease)
# are binary (0/1), not continuous. Plain SMOTE linearly interpolates between
# neighbors and would invent nonsensical values like 0.37 for a binary flag.
# SMOTENC treats specified columns as categorical and picks a real category
# value (via nearest-neighbor voting) instead of interpolating them.
from imblearn.over_sampling import SMOTENC

# ── Colab file upload (falls back gracefully outside Colab) ──────────────────
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("✅ All libraries imported successfully!")


## Step 2 — Upload & Load Dataset

In [ ]:
# Upload the CSV file (Colab) or point to a local path (outside Colab)
if IN_COLAB:
    uploaded = files.upload()          # Select ckd_1659.csv when prompted
    filename = list(uploaded.keys())[0]
else:
    filename = "ckd_1659.csv"          # ← edit this path if running locally

df_raw = pd.read_csv(filename)
print(f"✅ Dataset loaded: {filename}")
print(f"   Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")


## Step 3 — Dataset Exploration

In [ ]:
# ── Define the 11 features we'll use ─────────────────────────────────────────
FEATURES = [
    'Age', 'BMI', 'HbA1c', 'SerumCreatinine', 'BUNLevels',
    'GFR', 'HemoglobinLevels', 'CholesterolTotal',
    'ProteinInUrine', 'UrinaryTractInfections', 'FamilyHistoryKidneyDisease'
]
TARGET = 'Diagnosis'

print("=" * 55)
print(" DATASET OVERVIEW")
print("=" * 55)
print(f"\nTotal rows    : {df_raw.shape[0]}")
print(f"Total columns : {df_raw.shape[1]}")
print(f"Selected feat : {len(FEATURES)} features + 1 target")

# ── Data types of selected features ──────────────────────────────────────────
print("\n── Data Types of Selected Features ────────────────")
print(df_raw[FEATURES + [TARGET]].dtypes.to_string())

# ── Missing values ───────────────────────────────────────────────────────────
print("\n── Missing Values ──────────────────────────────────")
missing = df_raw[FEATURES + [TARGET]].isnull().sum()
print(missing.to_string())
print(f"\nTotal missing cells: {missing.sum()} (dataset is clean ✅)")

# ── Class distribution ───────────────────────────────────────────────────────
print("\n── Class Distribution (Target: Diagnosis) ──────────")
class_counts = df_raw[TARGET].value_counts()
print(f"  CKD    (1) : {class_counts[1]:>5}  ({class_counts[1]/len(df_raw)*100:.1f}%)")
print(f"  No CKD (0) : {class_counts[0]:>5}  ({class_counts[0]/len(df_raw)*100:.1f}%)")
print(f"  Imbalance ratio ≈ {class_counts[1]//class_counts[0]}:1")

# ── First 5 rows of selected features ────────────────────────────────────────
print("\n── Sample Rows (Selected Features) ─────────────────")
df_raw[FEATURES + [TARGET]].head()

## Step 4 — Feature Selection & Normalization

In [ ]:
# ── Keep only the 11 selected features + target ───────────────────────────────
df = df_raw[FEATURES + [TARGET]].copy()
print(f"Working dataframe shape: {df.shape}")

# ── Separate features (X) and target (y) ─────────────────────────────────────
X = df[FEATURES].values
y = df[TARGET].values

print("\nNOTE: Scaling is intentionally NOT done here. The scaler is fit on the")
print("training data only, *after* the train/test split (next step), so no")
print("information from the test set leaks into the transformation.")


## Step 5 — Stratified Train-Test Split (70 / 30)

In [ ]:
# stratify=y ensures both classes are proportionally represented in each split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y          # ← preserves the class ratio in both halves
)

# ── Fit the scaler on TRAINING data only, then apply it to both splits ───────
#    This avoids data leakage: no statistic from the test set influences scaling.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print("Train-Test Split Results")
print("-" * 40)
print(f"Training samples : {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Testing  samples : {len(X_test)}  ({len(X_test)/len(X)*100:.0f}%)")
print()
print("Class distribution in TRAINING set:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = 'CKD' if cls == 1 else 'No CKD'
    print(f"  {label} ({cls}): {cnt}")
print()
print("Class distribution in TEST set:")
unique, counts = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = 'CKD' if cls == 1 else 'No CKD'
    print(f"  {label} ({cls}): {cnt}")
print()
print("Scaling summary (mean ≈ 0, std ≈ 1, computed on TRAINING data only):")
print(pd.DataFrame(X_train, columns=FEATURES).describe().round(3).loc[['mean','std']])


## Step 6 — SMOTENC Oversampling (Training Data Only)


In [ ]:
# Record counts BEFORE SMOTENC
before_counts = dict(zip(*np.unique(y_train, return_counts=True)))

# Indices (within FEATURES) of the two binary/categorical columns.
# SMOTENC will pick an existing category (via neighbor voting) for these
# instead of linearly interpolating a fractional, meaningless value.
categorical_idx = [FEATURES.index('UrinaryTractInfections'),
                    FEATURES.index('FamilyHistoryKidneyDisease')]

# Apply SMOTENC only on training data — test set is NEVER touched
# sampling_strategy=1.0 → minority class will match majority class count
# k_neighbors=5  → use 5 nearest neighbors to synthesize new samples
smote = SMOTENC(categorical_features=categorical_idx,
                 sampling_strategy=1.0, k_neighbors=5, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Record counts AFTER SMOTENC
after_counts = dict(zip(*np.unique(y_train_sm, return_counts=True)))

print("SMOTENC Results (Training Data)")
print("-" * 40)
print(f"{'Class':<10} {'Before':>8} {'After':>8}")
print("-" * 40)
for cls in sorted(before_counts.keys()):
    label = f"CKD ({cls})"
    print(f"{label:<10} {before_counts[cls]:>8} {after_counts[cls]:>8}")
print("-" * 40)
print(f"{'Total':<10} {sum(before_counts.values()):>8} {sum(after_counts.values()):>8}")


## Step 7 — Visualize Class Distribution (Before vs After SMOTENC)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Distribution: Before vs After SMOTENC (Training Data)',
             fontsize=14, fontweight='bold', y=1.02)

labels = ['No CKD (0)', 'CKD (1)']
colors = ['#4CAF50', '#F44336']

# Before SMOTE
before_vals = [before_counts.get(0, 0), before_counts.get(1, 0)]
bars0 = axes[0].bar(labels, before_vals, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Before SMOTE', fontsize=13)
axes[0].set_ylabel('Sample Count')
axes[0].set_ylim(0, max(after_counts.values()) * 1.15)
for bar, val in zip(bars0, before_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', fontweight='bold')

# After SMOTE
after_vals = [after_counts.get(0, 0), after_counts.get(1, 0)]
bars1 = axes[1].bar(labels, after_vals, color=colors, edgecolor='black', width=0.5)
axes[1].set_title('After SMOTE', fontsize=13)
axes[1].set_ylabel('Sample Count')
axes[1].set_ylim(0, max(after_counts.values()) * 1.15)
for bar, val in zip(bars1, after_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
print("✅ SMOTENC has balanced the training classes (1:1 ratio), without inventing fractional values for the binary features.")

## Step 8 — Train All 4 Models

In [ ]:

lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_sm, y_train_sm)
print("✅ Logistic Regression trained.")

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_sm, y_train_sm)
print("✅ Random Forest trained.")


svm_model = SVC(
    kernel='rbf',
    class_weight='balanced',
    C=1.0,
    gamma='scale',
    random_state=42,
    probability=True
)
svm_model.fit(X_train_sm, y_train_sm)
print("✅ SVM (RBF kernel) trained.")

# SMOTE already balanced classes to 1:1 — no need for scale_pos_weight.

xgb_model = XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)
xgb_model.fit(X_train_sm, y_train_sm)
print("✅ XGBoost trained.")

## Step 8b — Isotonic Regression Calibration (XGBoost)

Platt scaling assumes the raw scores follow a logistic curve, which does not hold for boosted trees like XGBoost. Isotonic regression is non-parametric and fits the actual empirical calibration curve from out-of-fold predictions without any shape assumption, resulting in smoother probability transitions near the decision boundary, better-calibrated outputs for borderline patients, and more decisive probabilities at the extremes.

In [ ]:
# Isotonic is non-parametric and better suited than Platt sigmoid for
# XGBoost's step-like probability outputs.
calibrated_xgb = CalibratedClassifierCV(xgb_model, method='isotonic', cv=5)
calibrated_xgb.fit(X_train_sm, y_train_sm)
print("✅ XGBoost wrapped with isotonic regression calibration.")

## Step 9 — Evaluate All 4 Models

In [ ]:
# ── Predictions on the held-out test set ─────────────────────────────────────
lr_pred  = lr_model.predict(X_test)
rf_pred  = rf_model.predict(X_test)
svm_pred = svm_model.predict(X_test)
xgb_pred = calibrated_xgb.predict(X_test)

# ── Helper function: compute macro metrics AND per-class metrics ─────────────
#    Macro F1 alone can hide a model that is failing the minority class while
#    still doing well on the majority class — so we track class-0 ("No CKD",
#    the minority here) precision/recall explicitly and surface them later.
def get_metrics(y_true, y_pred, name):
    return {
        'Model'          : name,
        'Accuracy'       : accuracy_score(y_true, y_pred),
        'Precision'      : precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall'         : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'F1-Score'       : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'NoCKD_Precision': precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        'NoCKD_Recall'   : recall_score(y_true, y_pred, pos_label=0, zero_division=0),
    }

lr_metrics  = get_metrics(y_test, lr_pred,  'Logistic Regression')
rf_metrics  = get_metrics(y_test, rf_pred,  'Random Forest')
svm_metrics = get_metrics(y_test, svm_pred, 'SVM')
xgb_metrics = get_metrics(y_test, xgb_pred, 'XGBoost')

# ── Print all classification reports ─────────────────────────────────────────
for metrics, pred, name in [
    (lr_metrics,  lr_pred,  'LOGISTIC REGRESSION'),
    (rf_metrics,  rf_pred,  'RANDOM FOREST'),
    (svm_metrics, svm_pred, 'SVM'),
    (xgb_metrics, xgb_pred, 'XGBOOST'),
]:
    print("=" * 55)
    print(f" {name} — Classification Report")
    print("=" * 55)
    print(classification_report(y_test, pred, target_names=['No CKD (0)', 'CKD (1)']))


## Step 10 — Visualizations

In [ ]:
# ── 10a: Confusion Matrices — 2×2 grid ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Confusion Matrices — All 4 Models', fontsize=15, fontweight='bold')

model_info = [
    (lr_pred,  'Logistic Regression', 'Blues',   axes[0][0]),
    (rf_pred,  'Random Forest',       'Greens',  axes[0][1]),
    (svm_pred, 'SVM (RBF)',           'Oranges', axes[1][0]),
    (xgb_pred, 'XGBoost',            'Purples', axes[1][1]),
]

for pred, title, cmap, ax in model_info:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap=cmap,
        xticklabels=['No CKD (0)', 'CKD (1)'],
        yticklabels=['No CKD (0)', 'CKD (1)'],
        ax=ax, linewidths=0.5, linecolor='gray',
        annot_kws={'size': 13, 'weight': 'bold'}
    )
    ax.set_title(title, fontsize=13, fontweight='bold', pad=8)
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved as confusion_matrices.png")

In [ ]:
# ── 10b: Metrics comparison bar chart — 4 models ─────────────────────────────
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_results = [lr_metrics, rf_metrics, svm_metrics, xgb_metrics]
model_colors  = ['#2196F3', '#FF9800', '#E91E63', '#9C27B0']

x     = np.arange(len(metric_names))
width = 0.20   # narrower bars to fit 4 models

fig, ax = plt.subplots(figsize=(13, 6))

for i, (result, color) in enumerate(zip(model_results, model_colors)):
    vals = [result[m] for m in metric_names]
    bars = ax.bar(x + (i - 1.5) * width, vals, width,
                  label=result['Model'], color=color,
                  edgecolor='white', linewidth=0.8)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.008,
                f'{bar.get_height():.2f}',
                ha='center', fontsize=8, fontweight='bold')

ax.set_title('Model Performance Comparison — All 4 Models (Macro Average)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=12)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=10, loc='upper right')
ax.spines[['top','right']].set_visible(False)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved as metrics_comparison.png")

In [ ]:
# ── 10c: Feature Importance — Random Forest & XGBoost side by side ──────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Importance Comparison', fontsize=14, fontweight='bold')

# Random Forest
rf_imp = pd.DataFrame({'Feature': FEATURES, 'Importance': rf_model.feature_importances_})
rf_imp = rf_imp.sort_values('Importance', ascending=True)
colors_rf = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(rf_imp)))
bars = axes[0].barh(rf_imp['Feature'], rf_imp['Importance'],
                    color=colors_rf, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, rf_imp['Importance']):
    axes[0].text(val + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
axes[0].set_title('Random Forest', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Importance Score')
axes[0].spines[['top','right']].set_visible(False)
axes[0].xaxis.grid(True, alpha=0.3)

# XGBoost
xgb_imp = pd.DataFrame({'Feature': FEATURES, 'Importance': xgb_model.feature_importances_})
xgb_imp = xgb_imp.sort_values('Importance', ascending=True)
colors_xgb = plt.cm.RdYlBu(np.linspace(0.3, 0.9, len(xgb_imp)))
bars2 = axes[1].barh(xgb_imp['Feature'], xgb_imp['Importance'],
                     color=colors_xgb, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars2, xgb_imp['Importance']):
    axes[1].text(val + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
axes[1].set_title('XGBoost', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance Score')
axes[1].spines[['top','right']].set_visible(False)
axes[1].xaxis.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved as feature_importance.png")

## Step 11 — Final Summary & Best Model

In [ ]:
# ── Build the summary table ───────────────────────────────────────────────────
summary_df = pd.DataFrame([
    lr_metrics, rf_metrics, svm_metrics, xgb_metrics
]).set_index('Model').round(4)

print("=" * 65)
print(" FINAL MODEL COMPARISON SUMMARY (Macro Average)")
print("=" * 65)
print(summary_df[['Accuracy', 'Precision', 'Recall', 'F1-Score']].to_string())
print()
print("── Minority-class ('No CKD') detail ────────────────────────")
print("   High overall accuracy can hide poor minority-class detection —")
print("   these two columns show that directly, model by model:")
print(summary_df[['NoCKD_Precision', 'NoCKD_Recall']].to_string())
print()

# ── Declare the winner by F1-Score ────────────────────────────────────────────
best_model = summary_df['F1-Score'].idxmax()
best_row   = summary_df.loc[best_model]

print("=" * 65)
print(f" 🏆 BEST MODEL BY MACRO F1 : {best_model}")
print(f"    Accuracy        : {best_row['Accuracy']:.4f}")
print(f"    Precision       : {best_row['Precision']:.4f}")
print(f"    Recall          : {best_row['Recall']:.4f}")
print(f"    F1-Score        : {best_row['F1-Score']:.4f} (macro average)")
print(f"    No-CKD Recall   : {best_row['NoCKD_Recall']:.4f}  (of the {sum(y_test==0)} true No-CKD patients in the test set)")
print("=" * 65)
print()
print("Why F1-Score (macro) is the right *primary* metric here:")
print("  • Dataset has class imbalance (≈11:1).")
print("  • Macro average weights both classes equally, so a model can't")
print("    win just by predicting the majority class every time.")
print("  • F1 balances Precision and Recall — important when both false")
print("    positives AND false negatives carry a real clinical cost.")
print()
print("⚠️  Caveats worth keeping in mind:")
print(f"  • The test set has only {sum(y_test==0)} 'No CKD' patients. Metrics for that class")
print("    (and therefore the macro averages) can swing a lot with a different")
print("    random split — treat these numbers as noisy, not precise.")
print("  • Correlations between the 11 selected features and Diagnosis are")
print("    weak (all under 0.21 in this dataset), which caps how much any")
print("    model — regardless of tuning — can realistically separate the")
print("    classes. A high macro F1 here reflects the pipeline working")
print("    correctly, not necessarily strong real-world predictive power.")
print("  • Consider k-fold cross-validation instead of a single 70/30 split")
print("    before treating any one model as the definitive 'winner.'")
